In [0]:
dbutils.widgets.text('storage_account', '', '')
dbutils.widgets.text('key_vault_scope', '', '')

In [0]:
key_vault_scope = dbutils.widgets.get('key_vault_scope')  # pylint: disable=unused-variable

### Run shared notebook

In [0]:
%run /dxcore/Utilities/MailAlerts $key_vault_scope=key_vault_scope

In [0]:
#%run /dxcore/Utilities/great-expectations-init $storage_account=storage_account $key_vault_scope=key_vault_scope

In [0]:
storage_account = dbutils.widgets.get('storage_account')

In [0]:
%run /dxcore/Utilities/Utilities $storage_account=storage_account

### Define input parameters and load configuration

In [0]:
dbutils.widgets.text('ingestion_ts', '', '')
dbutils.widgets.text('bronze_partition', '', '')
dbutils.widgets.text('params', '', '')

bronze_partition  = dbutils.widgets.get('bronze_partition')
json_params        = dbutils.widgets.get('params')
#
# Parse JSON from input parameter
#
print(f"JSON params: {json_params}\n")

try:
  parsed_params = json.loads(json_params)
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f"Unable to parse input parameters, Error: {str(e)}"))
  
storage_account = parsed_params['SinkAzureStorageAccountName']
storage_account_uri = f"{storage_account}.dfs.core.windows.net" # pylint: disable=unused-variable 
  
#
# Get ingestion timestamp from a parameter passed into this notebook from ADF. If, for some reason, it has not been
# passed in successfully, set it to the current time.
#
table_name          = parsed_params['DataMovementShortName']
bronze_directory   ='/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])
silver_directory = '/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])
data_source_name = bronze_directory.split('/')[0]

try:
  ingestion_ts = dbutils.widgets.get('ingestion_ts')
  if not ingestion_ts:
    ingestion_ts = str(datetime.datetime.now())
  
  dataset = c.get_dataset_by_name(table_name)
  if not dataset:
    dataset = c.get_dataset_by_table_name(table_name)
  if not dataset:
    raise Exception(f"Unable to find configuration for dataset '{dataset_name}'")
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f'Error getting configuration, Error: {str(e)}'))
  

if not dataset:
  print(f'Dataset Name {dataset_name} is not found in the configuration file')
  dbutils.notebook.exit(get_error_payload(f'Dataset Name {dataset_name} is not found in the configuration file', ''))


print(f"Dataset {dataset.system}/{dataset.name} for table {dataset.table_name}")
print(f"bronze_partition = {bronze_partition}")
print(f"bronze_directory = {bronze_directory}")
print(f"silver_directory = {silver_directory}")
print(f"ingestion_ts = {ingestion_ts}")

In [0]:
def save_dataset_to_temp_table(data_frame: DataFrame, dataset: Dataset) -> None:
  temp_path = dataset.dataset_uri('temp', data_source_name)
  try:
    #
    # Persist the DataFrame. For better parallelization, first repartition into 60 partition, one for each Synapse distribution.
    #
    write_mode   = dataset.zones['temp'].databricks.mode   if dataset.zones['temp'].databricks.mode   else 'overwrite'
    write_format = dataset.zones['temp'].databricks.format if dataset.zones['temp'].databricks.format else 'parquet'
    #
    # Get optimal number of files into which dataset should be split
    #
    row_count = data_frame.count()
    partition_count = math.ceil(row_count / 1_000_000)
    if partition_count == 0:
      partition_count = 1
    #
    # Persist temp table
    #
    (data_frame
     .coalesce(partition_count)
     .write
     .mode(write_mode)
     .format(write_format)
     .save(temp_path)
    )
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(
      f"Unable to save temp table for Synapse Analytics at path {temp_path}, Error: {str(e)}", dataset.name))
    

###Get Bronze zone and silver zone config data

In [0]:
bronze_conf          = dataset.zones['bronze']
bronze_format        = bronze_conf.databricks.format
bronze_path          = dataset.dataset_uri(zone='bronze',dataset_name=bronze_directory)

silver_conf          = dataset.zones['silver']
silver_format        = silver_conf.databricks.format
silver_schema_main   = silver_conf.databricks.schema
silver_table_name    = dataset.datalake_table_name('silver')
silver_path          = dataset.dataset_uri('silver',dataset_name=silver_directory)
silver_write_mode    = dataset.mode

if dataset.partition_by and dataset.partition_by.get('silver'):
  silver_partition_col = dataset.partition_by.get('silver')
else:
  silver_partition_col = silver_conf.databricks.partition_by

print(f"Bronze format:\t\t\t{bronze_format}")
print(f"Bronze directory:\t\t{bronze_directory}")
print(f"Bronze path:\t\t\t{bronze_path}")
print(f"Silver format:\t\t\t{silver_format}")
print(f"Silver path:\t\t\t{silver_path}")

### Set the number of partitions Spark uses for shuffling to match the number of cores in the cluster

In [0]:
spark.conf.set('spark.sql.shuffle.partitions', sc.defaultParallelism)

###Some dates from the source system have unusual years (e.g. 0 or 1000). These dates can cause problems for Spark 3.0. To leave them unchanged, set settings to support legacy date format.

In [0]:
spark.conf.set('spark.sql.legacy.parquet.datetimeRebaseModeInRead', 'LEGACY')
spark.conf.set('spark.sql.legacy.parquet.datetimeRebaseModeInWrite', 'LEGACY')

### Enable caching and adaptive execution

In [0]:
spark.conf.set('spark.databricks.io.cache.enabled', 'true')
spark.conf.set('spark.sql.adaptive.enabled', 'true')

###Allow SQL MERGE to automatically update the schema in the Silver zone, if necessary

In [0]:
spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')

###Filter the dataset by job id and ingestion dt

In [0]:
try:
  ingestion_dt          = bronze_partition.split('/')[0].split('=')[1]
  job_id                = bronze_partition.split('/')[1].split('=')[1]
  is_fact = dataset.type.casefold() == 'fact'

  bronze_df = spark.read.format(bronze_format).load(bronze_path)
  if not is_fact:
    bronze_df = bronze_df.dropDuplicates()
  bronze_df = (bronze_df
                 .filter(F.col('ingestion_dt') == ingestion_dt).filter(F.col('job_id') == job_id))
  print(f"After filtering, {bronze_df.count():,} rows were loaded from bronze")
except Exception as e:
  dbutils.notebook.exit(get_error_payload(f"Error in filtering the dataset by job id and ingestion dt, Error: {str(e)}", dataset.name))

###Add hash_key to dimensional datasets

In [0]:
try:
  if dataset.hash_key:
#     print(f"Adding hash key {dataset.hash_key}")
    print(silver_table_name)
#     bronze_df = bronze_df.withColumn(dataset.hash_key, F.lit(None))
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(f"Error adding hash key column {dataset.hash_key}, Error: {str(e)}", dataset.name))
  
  

In [0]:
# add week and year columns

In [0]:
# if table_name.__contains__("tender"):
#   deltaTable = DeltaTable.forPath(spark, silver_path).delete("retailerId = 7889270188243873334")# pylint: disable=unused-variable

###Write the data in delta format in silver###

In [0]:
if silver_partition_col:
  #
  # Check whether it has a partition column (fact tables usually do) and partition by this column, if necessary
  #
  print(silver_partition_col)
  try:
    (bronze_df
     .write
     .format(silver_format)
     .mode(silver_write_mode)
     #      .option("mergeSchema", bronze_write_options)
     .partitionBy(silver_partition_col)
     .save(silver_path)
    )
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(
      f"Error persisting {dataset.name} from Bronze to Silver with partition column {silver_partition_col}, Error: {str(e)}", dataset.name))
    
else:
  #
  # If there is no partition column, persist to Silver wtihout a partition_by construct
  #
  try:
    (bronze_df
     .write
     .format(silver_format)
     .mode(silver_write_mode)
     #      .option("mergeSchema", bronze_write_options)
     .save(silver_path)
    )
  except Exception as e:
    
    dbutils.notebook.exit(get_error_payload(
      f"Error persisting {dataset.name} from Bronze to Silver with no partition column, Error: {str(e)}", dataset.name))
    

### Delete overlapping data for SPINS dataset

In [0]:
if table_name.__contains__("spins"):
  
  # To delete overlapping rows from SPINS dataset...

  # 2nd latest ingestion date
  dt = spark.sql('''
  SELECT CAST(max(ingestion_dt) as string) as dt
  from Delta.`{0}`
  where ingestion_dt != (SELECT max(ingestion_dt) from Delta.`{0}`)'''.format(silver_path))
  dt = dt.collect()[0][0]
  print(dt)

  # min time_period_end_date for latest ingestion date
  tp = spark.sql('''
  SELECT CAST(min(TIME_PERIOD_END_DATE) as string) as tp
  from Delta.`{0}`
  where ingestion_dt = (SELECT max(ingestion_dt) from Delta.`{0}`)'''.format(silver_path))
  tp = tp.collect()[0][0]
  print(tp)

  # Deletion of overlapping rows
  spark.sql('''DELETE FROM Delta.`{2}`
  WHERE ingestion_dt = '{0}' AND TIME_PERIOD_END_DATE >= '{1}' '''.format(dt,tp,silver_path))

### Create Silver database

In [0]:
#sc.setJobDescription('Creating Silver database')
if not silver_path.__contains__(dataset.name+"backfill"):
    try:
      create_db_in_metastore(
        schema_name=silver_schema_main
        ,location=dataset.database_uri('silver')
      )
    except Exception as e:
  
      dbutils.notebook.exit(get_error_payload(
        f"Unable to create db {silver_schema_main} in Databricks metastore, Error: {str(e)}", dataset.name))
  

In [0]:
if silver_table_name.__contains__("_archive"):
    new_silver_table_name = silver_table_name.replace("_archive", "")
else:
    new_silver_table_name = silver_table_name
print(new_silver_table_name)

In [0]:
#sc.setJobDescription(f'Surface {dataset.name} dataset as {silver_schema_main}.{silver_table_name}')

try:
  create_table_in_metastore(
    schema_name=silver_schema_main
    ,table_name=silver_table_name
    ,fmt=silver_format
    ,location=silver_path
  )
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Unable to create table {silver_schema_main}.{silver_table_name} in Databricks metastore, Error: {str(e)}", dataset.name))
  

###Write the data in parquet format with custom names in silver###

In [0]:
# Disabled "too-many-nested-blocks" Databricks warning for try block and it's message-id is R1702.
try:  # pylint: disable=R1702
  path = f"abfss://commercial-dx@{storage_account}.dfs.core.windows.net/silver/parquet/"+bronze_directory+"/"

  silver_df =  spark.read.format(silver_format).load(silver_path)
  silver_df = silver_df.dropDuplicates()
  silver_df = (silver_df
                 .filter(F.col('ingestion_dt') == ingestion_dt).filter(F.col('job_id') == job_id))
  print(f"After filtering, {silver_df.count():,} rows were loaded from silver")

  # run expectations on silver data frame
#   silver_df_ge = silver_df.limit(100)
#   result = get_ge_result(table_name,data_source_name,silver_df_ge,stage='bronze')
#   if result == 0:
#     print("No expectations created")
#   else:
#     assert result.success

  if bronze_path.__contains__("safegraph"):
  #   file_name = bronze_directory.split('/')[2]

    if bronze_path.__contains__("weekly"):
      no_of_lst_digits = 8
    elif bronze_path.__contains__("monthly"):
      no_of_lst_digits = 6

    if bronze_path.__contains__("core_poi-patterns"):
      silver_df = silver_df.withColumn('filepath_custom', regexp_replace('filepath','core_poi-patterns-part\d',''))  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', regexp_replace('filepath_custom','/\d',''))  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', regexp_replace('CustomName','\D','')).drop('filepath_custom')  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', when(length(col("CustomName"))>10,col('CustomName').substr(-no_of_lst_digits, no_of_lst_digits)).otherwise(col('CustomName')))
      silver_df.createOrReplaceTempView('silver_df')

    else:
      silver_df = silver_df.withColumn('CustomName', regexp_replace('filepath','\D',''))  # pylint: disable=W1401
      silver_df = silver_df.withColumn('CustomName', when(length(col("CustomName"))>10,col('CustomName').substr(-no_of_lst_digits, no_of_lst_digits)).otherwise(col('CustomName')))
      silver_df.createOrReplaceTempView('silver_df')

  elif bronze_path.__contains__("mapbox"):
  #   file_name = bronze_directory.split('/')[3]
    silver_df = silver_df.withColumn('CustomName', regexp_replace('agg_day_period','\D',''))  # pylint: disable=W1401
    silver_df = silver_df.withColumn('CustomName', when(length(col("CustomName"))>10,col('CustomName').substr(-6, 6)).otherwise(col('CustomName')))
    silver_df.createOrReplaceTempView('silver_df')

  elif bronze_path.__contains__("intelligent_sales"):
    silver_df.write.mode(silver_write_mode).parquet(path)

  elif bronze_path.__contains__("pdi"):
    silver_df.write.mode(silver_write_mode).parquet(path)

  elif bronze_path.__contains__("tdlinx"):
    silver_df.write.mode(silver_write_mode).parquet(path)
    
  elif bronze_path.__contains__("Skai"):
    silver_df.write.mode(silver_write_mode).parquet(path)
    
  elif bronze_path.__contains__("Skupos") or bronze_path.__contains__("skupos"):
    silver_df.write.mode(silver_write_mode).parquet(path)
  
  elif bronze_path.__contains__("Syndigo") or bronze_path.__contains__("syndigo"):
    silver_df.write.mode(silver_write_mode).parquet(path)
  
  elif bronze_path.__contains__("Dollar_General"):
    silver_df.write.mode(silver_write_mode).parquet(path)

  if bronze_path.__contains__("mapbox") or bronze_path.__contains__("safegraph"):
    if bronze_path.__contains__("brand_info"):
      df_all_names = spark.sql("""SELECT Max(CustomName) as CustomName FROM silver_df""")
      dbutils.fs.rm(path,True)
    else:
      df_all_names = spark.sql("""SELECT DISTINCT CustomName FROM silver_df where CustomName is not NULL""")

    l_all_names =  list(df_all_names.select("CustomName").rdd.flatMap(lambda x: x).collect())

    for name in l_all_names:
      df_custom_name = spark.sql("""SELECT * FROM silver_df WHERE CustomName = {0}""".format(name))
      df_custom_name = df_custom_name.drop('CustomName')
    #   out_file_name = file_name+'_'+name+'.snappy.parquet'  
      df_custom_name = df_custom_name.cache()
      df_custom_name.write.mode(silver_write_mode).parquet(path+name)
except Exception as e:
  dbutils.notebook.exit(get_error_payload(
    f"Error in writing data in {path}, Error: {str(e)}", dataset.name))

In [0]:
try:
  #
  # Count the number of rows in the bronze dataset
  #
  row_count = silver_df.count()
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Unable to filter dataset {dataset.name} ingested since {ingestion_ts}, Error: {str(e)}", dataset.name))
  
    
print(f"Loaded {row_count:,} rows from bronze")

###Populate temp table for Synapse Analytics load

In [0]:
try:  # pylint: disable=R1702
  if dataset.parquet_temp_table:
    if not is_fact:
      #
      # If this is a dimensional table, reload it in its entirety
      #
      silver_df = (spark
                   .read
                   .format(silver_format)
                   .load(silver_path)
                  )
#     else:
#       #
#       # If this is a fact table, use the incremental data from the Bronze zone
#       #
#       silver_df = bronze_df
    #
    # Persist the DataFrame to the temp table directory
    #
    save_dataset_to_temp_table(silver_df, dataset)
    print("Data saved in temp table")
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Unable to save temp table for Synapse Analytics at path {temp_path}, Error: {str(e)}", dataset.name))
  

###Returns success message to caller (in this case, Azure Data Factory)

In [0]:
dbutils.notebook.exit(json.dumps({
  "status": PipelineStatus.SUCCESS.value
  ,"dataset": dataset.name
  ,"row_count": row_count
  ,"partition_count": 1
  ,"is_translatable": dataset.is_translatable
  ,"error_msg": None
}))